# 03 · Baselines

Antes de qualquer rede neural, é preciso saber **o quanto métodos simples já resolvem**:

1. **Persistência:** a cota daqui a *h* horas = cota de agora.
2. **Regressão Ridge** (linear regularizada).
3. **Gradient boosting** (`HistGradientBoostingRegressor`).

Métrica-chave: `skill_vs_persist` (1 − MSE do modelo / MSE da persistência). **Só vale a pena se for > 0.**

In [ ]:
import os, sys
sys.path.append("..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, data, features, metrics, pipeline
plt.rcParams["figure.figsize"] = (11, 3.4)
plt.rcParams["axes.grid"] = True

from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

In [ ]:
panel = pipeline.load_panel()
resultados = {}

for alvo in config.TARGETS:
    if f"cota_{alvo}" not in panel:
        continue
    P = features.build_tabular(panel, alvo)
    tr, va, te = (P.masks[k] for k in ("train", "val", "test"))
    H = len(P.horizons)

    # 1) persistência: variação prevista = 0
    pred_abs = {"persistencia": np.repeat(P.base[:, None], H, axis=1)}

    # 2) Ridge (padroniza com o treino; buracos viram a média + indicador "faltando")
    Xn, _ = features.nn_inputs(P)
    ridge = Ridge(alpha=1.0).fit(Xn[tr], P.Y[tr])
    pred_abs["ridge"] = P.base[:, None] + ridge.predict(Xn)

    # 3) Gradient boosting: um modelo por horizonte (aceita NaN nativamente)
    delta = np.zeros_like(P.Y)
    for j in range(H):
        gb = HistGradientBoostingRegressor(max_depth=6, learning_rate=0.05, max_iter=300,
                                           early_stopping=False, random_state=config.RANDOM_STATE)
        gb.fit(P.X[tr], P.Y[tr][:, j])
        delta[:, j] = gb.predict(P.X)
    pred_abs["hgb"] = P.base[:, None] + delta

    for nome, pa in pred_abs.items():
        pipeline.save_predictions(nome, alvo, P, pa)

    for conj, m in (("validacao", va), ("teste", te)):
        if m.sum() == 0:
            continue
        for nome, pa in pred_abs.items():
            tab = metrics.summary_table(P.Yabs[m], pa[m], P.base[m], P.horizons)
            tab.insert(0, "modelo", nome); tab.insert(0, "conjunto", conj)
            resultados.setdefault(alvo, []).append(tab.reset_index())

    print(f"\n=== {config.STATIONS[alvo]['nome']} ({alvo}) ===")
    display(pd.concat(resultados[alvo]).set_index(["conjunto", "modelo", "horizonte_h"]))

In [ ]:
for alvo, tabs in resultados.items():
    pd.concat(tabs).to_csv(config.OUTPUTS / f"baselines_{alvo}.csv", index=False)
print("tabelas salvas em", config.OUTPUTS)

**Como ler:** `RMSE_cm`/`MAE_cm` em cm (menor é melhor). `NSE` fica alto até para a persistência (o nível é muito autocorrelacionado), então olhe o `skill_vs_persist`.
Se o gradient boosting ficar **perto da persistência** nos horizontes curtos, é um achado legítimo;